# 06 - Números do relatório

**Objetivo:** calcular, em um único lugar, cada número citado no relatório executivo, e gerar uma tabela de conferência com o valor calculado e o valor atual do relatório.

**Dependências:** rodar antes os notebooks 01 (gera `base_pedidos.csv`) e 05 (gera `metricas_modelo_nota_baixa.csv`).

**Definições de base** (declaradas aqui para todo o projeto):
- `validos`: pedidos do período, sem cancelados/indisponíveis e com itens. Usada em crescimento e recompra.
- `entregues`: pedidos do período com status *delivered* e data de entrega. Usada em logística.
- `df_nota`: entregues com nota de avaliação. Base única da satisfação, do atraso e do diagnóstico.

## 1. Configuração e bases

In [1]:
import math
import numpy as np
import pandas as pd
from pathlib import Path

from utils import br, pct, JANELA_INICIO as INICIO, JANELA_FIM as FIM

PROCESSED = Path("../data/processed")
RAW = Path("../data/raw")
TAB = Path("../reports/tabelas")
TAB.mkdir(parents=True, exist_ok=True)

DATAS = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
base = pd.read_csv(PROCESSED / "base_pedidos.csv", parse_dates=DATAS)
base["mes_compra"] = base["order_purchase_timestamp"].dt.to_period("M")

STATUS_EXCLUIDOS = ["canceled", "unavailable"]
ORDEM_ATRASO = [
    "Muito cedo (>5d antes)", "No prazo ou cedo (até 5d)", "Atraso leve (1 a 5d)",
    "Atraso moderado (5 a 10d)", "Atraso grave (>10d)",
]

janela = base[(base["mes_compra"] >= INICIO) & (base["mes_compra"] <= FIM)].copy()

validos = janela[~janela["order_status"].isin(STATUS_EXCLUIDOS) & janela["qtd_itens"].notna()].copy()
entregues = janela[
    (janela["order_status"] == "delivered")
    & janela["order_delivered_customer_date"].notna()
    & ~janela["excluir_logistica"]
].copy()
df_nota = entregues[entregues["review_score"].notna()].copy()

registro = []
def registra(numero, calculado, no_relatorio, onde):
    registro.append({"numero": numero, "calculado": calculado,
                     "no_relatorio_atual": no_relatorio, "onde": onde})

print(f"Pedidos no período:              {br(len(janela))}")
print(f"Pedidos válidos (crescimento):   {br(len(validos))}")
print(f"Entregues no período:            {br(len(entregues))}")
print(f"Entregues com nota (base única): {br(len(df_nota))}")

Pedidos no período:              99.092
Pedidos válidos (crescimento):   97.905
Entregues no período:            94.834
Entregues com nota (base única): 94.198


## 2. Crescimento e receita (seções 2 e 3)

In [2]:
mensal = validos.groupby("mes_compra").agg(
    pedidos=("order_id", "count"), gmv=("receita_produtos", "sum"),
)

def jan_ago(ano):
    m = mensal.loc[f"{ano}-01":f"{ano}-08"]
    return m["pedidos"].sum(), m["gmv"].sum()

p17, g17 = jan_ago(2017)
p18, g18 = jan_ago(2018)
registra("Crescimento de pedidos jan-ago", f"{(p18/p17-1)*100:+.0f}%", "137%", "Sumário; seção 3")
registra("Crescimento de GMV jan-ago", f"{(g18/g17-1)*100:+.0f}%", "138%", "Sumário; seção 3")
registra("Ticket médio jan-ago 2017 (R$)", br(g17/p17, 2), "136,55", "Seção 3")
registra("Ticket médio jan-ago 2018 (R$)", br(g18/p18, 2), "137,14", "Seção 3")

itens = pd.read_csv(RAW / "olist_order_items_dataset.csv", usecols=["order_id", "product_id", "price"])
itens = itens[itens["order_id"].isin(validos["order_id"])]
produtos = pd.read_csv(RAW / "olist_products_dataset.csv", usecols=["product_id", "product_category_name"])
trad = pd.read_csv(RAW / "product_category_name_translation.csv")
itens = itens.merge(produtos, on="product_id", how="left").merge(trad, on="product_category_name", how="left")
itens["categoria"] = (itens["product_category_name_english"]
                      .fillna(itens["product_category_name"]).fillna("sem_categoria"))

cat = itens.groupby("categoria")["price"].sum().sort_values(ascending=False)
share_cat = cat / cat.sum()
n80 = int((share_cat.cumsum() < 0.80).sum() + 1)
registra("Categorias que somam 80% do GMV", f"{n80} de {len(cat)}", "18 de 74", "Seção 3")
registra("Top 3 categorias (% GMV)", pct(share_cat.head(3).sum()), "26%", "Seção 3")

uf = validos.groupby("customer_state").agg(
    pedidos=("order_id", "count"), gmv=("receita_produtos", "sum"), frete=("frete", "sum"),
)
uf["part"] = uf["pedidos"] / uf["pedidos"].sum()
uf["frete_pct"] = uf["frete"] / uf["gmv"]
registra("Top 3 estados (% pedidos)",
         pct(uf.sort_values("pedidos", ascending=False)["part"].head(3).sum()), "67%", "Seção 3")
registra("São Paulo (% pedidos)", pct(uf.loc["SP", "part"]), "42%", "Seção 3")
for estado, rel in {"SP": "14%", "RJ": "17%", "MG": "17%", "BA": "20%"}.items():
    registra(f"Frete como % do GMV: {estado}", pct(uf.loc[estado, "frete_pct"]), rel, "Seção 3")

## 3. Logística e atraso (seções 4 e 9)

In [3]:
lt = entregues[["dias_aprovacao", "dias_postagem", "dias_transporte", "lead_time_total"]].mean()
registra("Lead time total (dias)", br(lt["lead_time_total"], 1), "12,5", "Sumário; seção 4")
registra("Aprovação (dias)", br(lt["dias_aprovacao"], 1), "0,4", "Seção 4")
registra("Postagem (dias)", br(lt["dias_postagem"], 1), "2,8", "Seção 4")
registra("Transporte (dias)", br(lt["dias_transporte"], 1), "9,3", "Seção 4")

dn = df_nota.copy()
dn["faixa"] = pd.cut(dn["dias_atraso"], bins=[-np.inf, -5, 0, 5, 10, np.inf], labels=ORDEM_ATRASO)
atrasados_n = int((dn["dias_atraso"] > 0).sum())
dist = dn["faixa"].value_counts().reindex(ORDEM_ATRASO)

assert dist.iloc[2:].sum() == atrasados_n, "Faixas e flag de atraso divergem!"

registra("No prazo ou antes (% da base)", pct(1 - atrasados_n / len(dn), 1), "92%", "Sumário; seção 4")
registra("Atrasados (n)", br(atrasados_n), "7.822 (a corrigir)", "Sumário; seções 4 e 7")
registra("Atrasados (% da base)", pct(atrasados_n / len(dn), 1), "8,1% (a corrigir)", "Sumário; seção 4")

share_atr = dist.iloc[2:] / atrasados_n
registra("Atrasados de 1 a 5 dias (% dos atrasados)", pct(share_atr.iloc[0]), "46%", "Seção 4")
registra("Atrasados de 5 a 10 dias (% dos atrasados)", pct(share_atr.iloc[1]), "24%", "Seção 4")
registra("Atrasados acima de 10 dias (% dos atrasados)", pct(share_atr.iloc[2]), "29%", "Seção 4")

nota_faixa = dn.groupby("faixa", observed=True)["review_score"].mean()
sat_faixa = dn.assign(sat=dn["review_score"] >= 4).groupby("faixa", observed=True)["sat"].mean()
registra("Nota média, atraso de 1 a 5 dias", br(nota_faixa["Atraso leve (1 a 5d)"], 2), "3,46", "Seção 4")
registra("Satisfeitos, atraso de 1 a 5 dias", pct(sat_faixa["Atraso leve (1 a 5d)"]), "57%", "Seção 4")
registra("Nota média, atraso de 5 a 10 dias", br(nota_faixa["Atraso moderado (5 a 10d)"], 2), "1,89", "Seção 4")
registra("Satisfeitos, atraso de 5 a 10 dias", pct(sat_faixa["Atraso moderado (5 a 10d)"]), "17%", "Seção 4")
registra("Nota média, atraso acima de 10 dias", br(nota_faixa["Atraso grave (>10d)"], 2), "1,70", "Seção 4; sumário")
registra("Satisfeitos, atraso acima de 10 dias", pct(sat_faixa["Atraso grave (>10d)"]), "11%", "Seção 4")
registra("Correlação atraso x nota", br(dn["dias_atraso"].corr(dn["review_score"]), 2), "-0,27", "Notebook 03")

por_uf = dn.groupby("customer_state").agg(
    pedidos=("order_id", "count"), atrasados=("atrasado", "sum"),
)
por_uf["taxa"] = por_uf["atrasados"] / por_uf["pedidos"]
por_uf["atrasados"] = por_uf["atrasados"].astype(int)

assert por_uf["atrasados"].sum() == atrasados_n, "Total por estado não bate com o total geral!"

rj = por_uf.loc["RJ", "atrasados"]
registra("Atrasados no RJ (n)", br(rj), "1.660", "Seções 4 e 9")
registra("RJ (% do total de atrasos)", pct(rj / atrasados_n), "cerca de 1/5", "Seção 4")
quatro = ["AL", "MA", "PI", "CE"]
registra("AL+MA+PI+CE atrasados (n)", br(por_uf.loc[quatro, "atrasados"].sum()), "cerca de 500", "Seções 4 e 9")
registra("AL+MA+PI+CE (% do total)", pct(por_uf.loc[quatro, "atrasados"].sum() / atrasados_n, 1), "6,5%", "Seção 4")
for estado, rel in {"AL": "24%", "MA": "20%", "PI": "16%", "CE": "15%"}.items():
    registra(f"Taxa de atraso {estado}", pct(por_uf.loc[estado, "taxa"]), rel, "Seção 4")

## 4. Recompra (seção 5)

In [4]:
pedidos_cliente = validos.groupby("customer_unique_id").size()
recomprou_flag = pedidos_cliente > 1
registra("Clientes que recompraram", pct(recomprou_flag.mean(), 1), "3,1%", "Sumário; seção 5")
registra("Clientes que compraram uma vez", pct(1 - recomprou_flag.mean(), 1), "96,9%", "Seção 5")

# Comparação com a definição antiga (todos os status), para documentar a diferença
todos = janela.groupby("customer_unique_id").size() > 1
print(f"Comparação: recompra com todos os status = {pct(todos.mean(), 1)}")

pc = (validos.sort_values("order_purchase_timestamp")
      .drop_duplicates("customer_unique_id", keep="first").copy())
pc["recomprou"] = pc["customer_unique_id"].map(recomprou_flag).astype(bool)
pc_nota = pc[pc["review_score"].notna()].copy()

pc_nota["grupo_nota"] = np.where(pc_nota["review_score"] >= 4, "4-5", "1-3")
g_nota = pc_nota.groupby("grupo_nota")["recomprou"].agg(["mean", "count"])
registra("Recompra, nota 4-5 na 1ª compra", pct(g_nota.loc["4-5", "mean"], 1), "3,1%", "Seção 5")
registra("Recompra, nota 1-3 na 1ª compra", pct(g_nota.loc["1-3", "mean"], 1), "3,1%", "Seção 5")
diff_pp = (g_nota.loc["4-5", "mean"] - g_nota.loc["1-3", "mean"]) * 100
registra("Diferença de recompra por nota (pp)", br(diff_pp, 1), "0,0", "Sumário; seção 5")

grave = pc_nota[pc_nota["dias_atraso"] > 5]
resto = pc_nota[pc_nota["dias_atraso"] <= 5]
p1, n1 = grave["recomprou"].mean(), len(grave)
p2, n2 = resto["recomprou"].mean(), len(resto)
pool = (grave["recomprou"].sum() + resto["recomprou"].sum()) / (n1 + n2)
z = (p1 - p2) / math.sqrt(pool * (1 - pool) * (1 / n1 + 1 / n2))
p_valor = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))

registra("Recompra, atraso acima de 5 dias", pct(p1, 1), "2,6%", "Seção 5")
registra("Recompra, demais pedidos", pct(p2, 1), "3,2%", "Seção 5")
registra("Teste z: diferença atraso > 5d", f"z = {br(z, 2)}, p = {br(p_valor, 3)}", "não testado", "Seção 5 (a incluir)")
registra("Nota média de quem atrasou mais de 5 dias", br(grave["review_score"].mean(), 2), "1,8", "Seção 5")

def coorte(m):
    if m <= pd.Period("2017-06"):
        return "2017 jan-jun"
    if m <= pd.Period("2017-12"):
        return "2017 jul-dez"
    return "2018 jan-ago"

pc["coorte"] = pc["mes_compra"].apply(coorte)
coortes = pc.groupby("coorte")["recomprou"].mean()
registra("Recompra coorte 2017 jan-jun", pct(coortes["2017 jan-jun"], 1), "5,2%", "Seção 5")
registra("Recompra coorte 2017 jul-dez", pct(coortes["2017 jul-dez"], 1), "3,8%", "Seção 5")
registra("Recompra coorte 2018 jan-ago", pct(coortes["2018 jan-ago"], 1), "2,1%", "Seção 5")

Comparação: recompra com todos os status = 3,1%


## 5. Diagnóstico do modelo (seção 6)

Lê as métricas salvas pelo notebook 05. Rode o 05 antes deste notebook.

In [5]:
metricas = pd.read_csv(TAB / "metricas_modelo_nota_baixa.csv").set_index("Metrica")["Valor"]

registra("Acurácia do modelo", br(metricas["Acurácia"], 2), "0,88", "Seção 6")
registra("ROC-AUC do modelo", br(metricas["ROC-AUC"], 2), "0,72", "Seção 6")
registra("Recall de nota baixa", pct(metricas["Recall (nota baixa)"]), "14%", "Seção 6")
registra("Precisão de nota baixa", pct(metricas["Precisão (nota baixa)"]), "72%", "Seção 6")

taxa_baixa = (dn["review_score"] <= 2).mean()
registra("Pedidos com nota baixa (1-2)", pct(taxa_baixa, 1), "12,8%", "Seção 6")
registra("Acerto de 'sempre nota OK'", pct(1 - taxa_baixa), "87%", "Seção 6")
registra("Pedidos avaliados no modelo", br(len(dn)), "95.568 (a corrigir)", "Seção 6")

## 6. Tabela de conferência

Salva `numeros_relatorio.csv` e destaca os números que diferem do relatório atual.

In [6]:
reg = pd.DataFrame(registro)
reg.to_csv(TAB / "numeros_relatorio.csv", index=False)

reg["muda"] = reg["calculado"] != reg["no_relatorio_atual"]
print(f"{reg['muda'].sum()} de {len(reg)} números diferem do relatório atual.\n")
print(reg.to_string(index=False))

28 de 56 números diferem do relatório atual.

                                      numero            calculado  no_relatorio_atual                  onde  muda
              Crescimento de pedidos jan-ago                +137%                137%      Sumário; seção 3  True
                  Crescimento de GMV jan-ago                +138%                138%      Sumário; seção 3  True
              Ticket médio jan-ago 2017 (R$)               136,55              136,55               Seção 3 False
              Ticket médio jan-ago 2018 (R$)               137,14              137,14               Seção 3 False
             Categorias que somam 80% do GMV             18 de 74            18 de 74               Seção 3 False
                    Top 3 categorias (% GMV)                  26%                 26%               Seção 3 False
                   Top 3 estados (% pedidos)                  67%                 67%               Seção 3 False
                       São Paulo (% pedido